# 02 - pandas ile Veri Analizi ve Veri Ön İşleme
**Veri Madenciliği Dersi – Gümüşhane Üniversitesi, Yönetim Bilişim Sistemleri**

Veri madenciliği projelerinde zamanın büyük kısmı (%60–80) verinin **okunması, temizlenmesi ve dönüştürülmesi** ile geçer. pandas bu işlerin ana aracıdır.

### Öğrenme Hedefleri
1. `Series` ve `DataFrame` yapılarını tanımak
2. Veriyi okumak, ilk keşifsel incelemeyi yapmak (`head`, `info`, `describe`)
3. `loc` / `iloc` ve koşullu filtreleme ile veri seçmek
4. **Eksik veri, tekrar eden kayıt ve hatalı tip** sorunlarını çözmek
5. Yeni sütun türetmek, tarihlerle çalışmak
6. `groupby`, `pivot_table` ve `merge` ile özetleme ve birleştirme

> Bu defterde kurgusal bir **e-ticaret satış veri seti** kullanacağız. Veri, defterin içinde üretildiği için ek dosyaya ihtiyaç yoktur.

**Kullanım:** `✍️ Alıştırma` hücrelerini kendiniz doldurun. Çözümler en sondadır.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
print("pandas sürümü:", pd.__version__)

pandas sürümü: 3.0.2


## 1. Series: Etiketli Tek Boyutlu Veri
`Series`, NumPy dizisine **indeks (etiket)** eklenmiş halidir.

In [2]:
nufus = pd.Series(
    {"İstanbul": 15.7, "Ankara": 5.8, "İzmir": 4.5, "Trabzon": 0.82, "Gümüşhane": 0.14},
    name="Nüfus (milyon)"
)
print(nufus)
print("\nAnkara:", nufus["Ankara"])
print("1 milyondan büyük iller:\n", nufus[nufus > 1])

İstanbul     15.70
Ankara        5.80
İzmir         4.50
Trabzon       0.82
Gümüşhane     0.14
Name: Nüfus (milyon), dtype: float64

Ankara: 5.8
1 milyondan büyük iller:
 İstanbul    15.7
Ankara       5.8
İzmir        4.5
Name: Nüfus (milyon), dtype: float64


In [ ]:
print("Ortalama:", nufus.mean().round(2))
print("Sıralı:\n", nufus.sort_values(ascending=False))

## 2. DataFrame Oluşturma
`DataFrame` bir Excel tablosu gibidir: satırlar **gözlemler**, sütunlar **değişkenler (özellikler)**. Her sütun bir `Series`'tir.

In [3]:
ogrenciler = pd.DataFrame({
    "ad":      ["Ayşe", "Mehmet", "Zeynep", "Emre", "Elif"],
    "bolum":   ["YBS", "İşletme", "YBS", "İktisat", "YBS"],
    "vize":    [70, 55, 92, 65, 80],
    "final":   [85, 48, 95, 72, 60],
})
ogrenciler

,ad,bolum,vize,final
0,Ayşe,YBS,70,85
1,Mehmet,İşletme,55,48
2,Zeynep,YBS,92,95
3,Emre,İktisat,65,72
4,Elif,YBS,80,60


### Veri setimiz: E-ticaret siparişleri
Aşağıdaki hücre, bilerek **kirli** (eksik değerli, tekrar eden, hatalı) bir veri seti üretir. Gerçek hayatta veriyi genellikle `pd.read_csv("dosya.csv")` veya `pd.read_excel("dosya.xlsx")` ile okursunuz.

In [4]:
rng = np.random.default_rng(42)
n = 300

urunler = {
    "Laptop": ("Elektronik", 32000), "Kulaklık": ("Elektronik", 1500),
    "Telefon": ("Elektronik", 25000), "Mont": ("Giyim", 3200),
    "Ayakkabı": ("Giyim", 2400), "Tişört": ("Giyim", 450),
    "Kitap": ("Kitap", 220), "Defter": ("Kırtasiye", 60),
    "Kalem Seti": ("Kırtasiye", 180), "Kahve Makinesi": ("Ev", 4800),
}
iller = ["İstanbul", "Ankara", "İzmir", "Trabzon", "Gümüşhane", "Erzurum", "Bursa"]

urun_sec = rng.choice(list(urunler), size=n)
df = pd.DataFrame({
    "siparis_id": np.arange(1001, 1001 + n),
    "tarih": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "musteri_id": rng.integers(1, 81, n),
    "il": rng.choice(iller, size=n, p=[0.3, 0.2, 0.15, 0.1, 0.05, 0.1, 0.1]),
    "urun": urun_sec,
    "kategori": [urunler[u][0] for u in urun_sec],
    "birim_fiyat": [urunler[u][1] * rng.uniform(0.9, 1.1) for u in urun_sec],
    "adet": rng.integers(1, 5, n),
    "odeme": rng.choice(["Kredi Kartı", "Havale", "Kapıda Ödeme"], size=n, p=[0.65, 0.2, 0.15]),
    "puan": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float),
})
df["birim_fiyat"] = df["birim_fiyat"].round(2)

# --- Veriyi bilerek kirletiyoruz ---
df.loc[rng.choice(n, 25, replace=False), "puan"] = np.nan          # eksik puanlar
df.loc[rng.choice(n, 8, replace=False), "il"] = np.nan             # eksik iller
df.loc[rng.choice(n, 3, replace=False), "adet"] = -1               # hatalı adet
df.loc[[5, 17], "il"] = ["  ankara ", "İSTANBUL"]                  # yazım tutarsızlığı
df = pd.concat([df, df.iloc[[10, 20, 30, 40]]], ignore_index=True) # tekrar eden satırlar

print("Veri seti hazır:", df.shape)

Veri seti hazır: (304, 10)


In [5]:
df

,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan
0,1001,2025-05-13,50,İstanbul,Laptop,Elektronik,30466.23,4,Kredi Kartı,5.0
1,1002,2025-10-29,63,Erzurum,Defter,Kırtasiye,57.87,3,Kredi Kartı,5.0
2,1003,2025-01-04,60,Trabzon,Kitap,Kitap,208.67,1,Havale,4.0
3,1004,2025-11-24,11,İstanbul,Ayakkabı,Giyim,2390.33,1,Havale,4.0
4,1005,2025-07-18,67,İstanbul,Ayakkabı,Giyim,2487.96,4,Kredi Kartı,5.0
...,...,...,...,...,...,...,...,...,...,...
299,1300,2025-08-23,50,Erzurum,Laptop,Elektronik,30378.75,3,Havale,4.0
300,1011,2025-09-01,67,İzmir,Tişört,Giyim,469.04,2,Havale,3.0
301,1021,2025-04-28,35,Ankara,Tişört,Giyim,426.75,4,Kredi Kartı,3.0
302,1031,2025-07-19,59,Ankara,Ayakkabı,Giyim,2282.05,3,Kredi Kartı,4.0


## 3. İlk Keşif (Keşifsel Veri Analizinin İlk Adımı) ⭐
Yeni bir veri setiyle karşılaştığınızda **her zaman** şu komutlarla başlayın:

In [6]:
df.head()        # ilk 5 satır (tail() son satırlar, sample() rastgele satırlar)

,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan
0,1001,2025-05-13,50,İstanbul,Laptop,Elektronik,30466.23,4,Kredi Kartı,5.0
1,1002,2025-10-29,63,Erzurum,Defter,Kırtasiye,57.87,3,Kredi Kartı,5.0
2,1003,2025-01-04,60,Trabzon,Kitap,Kitap,208.67,1,Havale,4.0
3,1004,2025-11-24,11,İstanbul,Ayakkabı,Giyim,2390.33,1,Havale,4.0
4,1005,2025-07-18,67,İstanbul,Ayakkabı,Giyim,2487.96,4,Kredi Kartı,5.0


In [7]:
df.info()        # sütun tipleri, eksik olmayan değer sayısı, bellek

<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   siparis_id   304 non-null    int32         
 1   tarih        304 non-null    datetime64[us]
 2   musteri_id   304 non-null    int64         
 3   il           296 non-null    str           
 4   urun         304 non-null    str           
 5   kategori     304 non-null    str           
 6   birim_fiyat  304 non-null    float64       
 7   adet         304 non-null    int64         
 8   odeme        304 non-null    str           
 9   puan         279 non-null    float64       
dtypes: datetime64[us](1), float64(2), int32(1), int64(2), str(4)
memory usage: 32.7 KB


In [8]:
df.describe()    # sayısal sütunların özet istatistikleri

,siparis_id,tarih,musteri_id,birim_fiyat,adet,puan
count,304.000000,304,304.000000,304.000000,304.000000,279.000000
mean,1148.861842,2025-07-02 20:17:22.105263,41.773026,6300.496743,2.467105,3.967742
min,1001.000000,2025-01-02 00:00:00,1.000000,54.060000,-1.000000,1.000000
25%,1072.750000,2025-03-27 12:00:00,20.000000,224.212500,1.750000,3.000000
50%,1148.500000,2025-07-04 00:00:00,44.000000,1619.020000,2.000000,4.000000
75%,1224.250000,2025-10-17 06:00:00,62.000000,3717.640000,4.000000,5.000000
max,1300.000000,2025-12-31 00:00:00,80.000000,34900.520000,4.000000,5.000000
std,87.345530,NaN,23.657299,10659.875816,1.165612,1.019080


In [9]:
df.describe(exclude="number")   # sayısal olmayan (metin, tarih) sütunların özeti

,tarih,il,urun,kategori,odeme
count,304,296,304,304,304
unique,NaN,9,10,5,3
top,NaN,İstanbul,Ayakkabı,Giyim,Kredi Kartı
freq,NaN,81,49,103,207
mean,2025-07-02 20:17:22.105263,NaN,NaN,NaN,NaN
min,2025-01-02 00:00:00,NaN,NaN,NaN,NaN
25%,2025-03-27 12:00:00,NaN,NaN,NaN,NaN
50%,2025-07-04 00:00:00,NaN,NaN,NaN,NaN
75%,2025-10-17 06:00:00,NaN,NaN,NaN,NaN
max,2025-12-31 00:00:00,NaN,NaN,NaN,NaN


In [10]:
print("Şekil:", df.shape)
print("Sütunlar:", list(df.columns))
print("\nKategori dağılımı:")
print(df["kategori"].value_counts())
print("\nÖdeme yöntemi oranları:")
print(df["odeme"].value_counts(normalize=True).round(3))
print("\nBenzersiz il sayısı:", df["il"].nunique())

Şekil: (304, 10)
Sütunlar: ['siparis_id', 'tarih', 'musteri_id', 'il', 'urun', 'kategori', 'birim_fiyat', 'adet', 'odeme', 'puan']

Kategori dağılımı:
kategori
Giyim         103
Elektronik     84
Kırtasiye      62
Kitap          32
Ev             23
Name: count, dtype: int64

Ödeme yöntemi oranları:
odeme
Kredi Kartı     0.681
Kapıda Ödeme    0.178
Havale          0.141
Name: proportion, dtype: float64

Benzersiz il sayısı: 9


> 🔎 **Gözlem soruları:** `describe()` çıktısında `adet` sütununun minimum değerine bakın — mantıklı mı? `info()` çıktısında hangi sütunlarda eksik değer var? Bu soruları 6. bölümde çözeceğiz.

## 4. Sütun ve Satır Seçimi: `[]`, `loc`, `iloc` ⭐
- `df["sütun"]` → tek sütun (Series)  |  `df[["s1", "s2"]]` → birden çok sütun (DataFrame)
- `loc` → **etiket** ile seçim: `df.loc[satır_etiketi, sütun_adı]` (dilimde bitiş **dahil**)
- `iloc` → **konum** ile seçim: `df.iloc[satır_no, sütun_no]` (dilimde bitiş **dahil değil**)

In [11]:
df[["urun", "birim_fiyat", "adet"]].head(3)

,urun,birim_fiyat,adet
0,Laptop,30466.23,4
1,Defter,57.87,3
2,Kitap,208.67,1


In [12]:
print(df.loc[0:3, ["urun", "il"]])      # 0,1,2,3 -> 4 satır
print()
print(df.iloc[0:3, 0:4])                # 0,1,2 -> 3 satır, ilk 4 sütun

       urun        il
0    Laptop  İstanbul
1    Defter   Erzurum
2     Kitap   Trabzon
3  Ayakkabı  İstanbul

   siparis_id      tarih  musteri_id        il
0        1001 2025-05-13          50  İstanbul
1        1002 2025-10-29          63   Erzurum
2        1003 2025-01-04          60   Trabzon


## 5. Koşullu Filtreleme
NumPy'daki boolean maskeleme mantığı aynen geçerlidir: `&`, `|`, `~` ve **parantezler**.

In [13]:
# Trabzon'dan verilen elektronik siparişler
df[(df["il"] == "Trabzon") & (df["kategori"] == "Elektronik")].head()

,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan
40,1041,2025-08-20,54,Trabzon,Kulaklık,Elektronik,1414.91,3,Kapıda Ödeme,5.0
146,1147,2025-04-25,43,Trabzon,Laptop,Elektronik,30579.93,3,Kredi Kartı,3.0
186,1187,2025-05-02,3,Trabzon,Kulaklık,Elektronik,1632.46,4,Havale,4.0
215,1216,2025-02-08,67,Trabzon,Telefon,Elektronik,24910.15,1,Havale,5.0
223,1224,2025-10-29,53,Trabzon,Telefon,Elektronik,27084.24,3,Kredi Kartı,4.0


In [14]:
# isin: birden fazla değerden biri
karadeniz = df[df["il"].isin(["Trabzon", "Gümüşhane"])]
print("Karadeniz siparişleri:", len(karadeniz))

# query: daha okunabilir alternatif
df.query("birim_fiyat > 20000 and adet >= 3").head()

Karadeniz siparişleri: 44


,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan
0,1001,2025-05-13,50,İstanbul,Laptop,Elektronik,30466.23,4,Kredi Kartı,5.0
6,1007,2025-04-21,26,Bursa,Laptop,Elektronik,30916.71,4,Kredi Kartı,5.0
31,1032,2025-05-16,27,İstanbul,Telefon,Elektronik,27093.97,4,Kredi Kartı,2.0
32,1033,2025-04-24,35,İzmir,Laptop,Elektronik,31708.53,3,Kredi Kartı,4.0
35,1036,2025-10-23,55,NaN,Laptop,Elektronik,31113.83,3,Kredi Kartı,4.0


In [15]:
# Metin işlemleri: .str erişimcisi
df[df["urun"].str.contains("Kahve")].head(3)

,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan
11,1012,2025-09-03,38,İstanbul,Kahve Makinesi,Ev,4464.44,3,Havale,3.0
23,1024,2025-12-01,34,İstanbul,Kahve Makinesi,Ev,5135.46,3,Kredi Kartı,5.0
45,1046,2025-04-04,27,İzmir,Kahve Makinesi,Ev,4652.62,2,Kredi Kartı,4.0


### ✍️ Alıştırma 1
1. Sadece `urun`, `kategori` ve `puan` sütunlarını içeren ilk 5 satırı gösterin.
2. Kapıda ödeme ile yapılmış ve puanı 2 veya altında olan siparişlerin sayısını bulun.
3. Giyim veya Kitap kategorisinde olup birim fiyatı 1000 TL'den düşük siparişleri listeleyin.

In [16]:
# Kodunuzu buraya yazın

## 6. Veri Temizleme ⭐⭐
Veri madenciliğinde **"çöp girerse çöp çıkar" (garbage in, garbage out)** kuralı geçerlidir. Modelleme öncesi bu adımlar mutlaka yapılmalıdır.

### 6.1 Tekrar eden kayıtlar

In [17]:
print("Tekrar eden satır sayısı:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Temizlik sonrası şekil:", df.shape)

Tekrar eden satır sayısı: 4
Temizlik sonrası şekil: (300, 10)


### 6.2 Tutarsız metin değerleri

In [18]:
print("Önce:", df["il"].unique())

df["il"] = df["il"].str.strip().str.title()
df["il"] = df["il"].replace({"Istanbul": "İstanbul"})   # Python'un title() fonksiyonu Türkçe İ harfini bilmez!

print("Sonra:", df["il"].unique())

Önce: <ArrowStringArray>
[ 'İstanbul',   'Erzurum',   'Trabzon', '  ankara ',     'Bursa',    'Ankara',
 'Gümüşhane',     'İzmir',  'İSTANBUL',         nan]
Length: 10, dtype: str
Sonra: <ArrowStringArray>
['İstanbul', 'Erzurum', 'Trabzon', 'Ankara', 'Bursa', 'Gümüşhane', 'İzmir',
 nan]
Length: 8, dtype: str


> ⚠️ **Türkçe karakter tuzağı:** `"İSTANBUL".lower()` sonucu `"i̇stanbul"` olur (noktalı i + birleşik nokta). Türkçe metinlerde büyük/küçük harf dönüşümlerinden sonra sonucu mutlaka kontrol edin.

### 6.3 Mantıksız (hatalı) değerler

In [19]:
print("Hatalı adet içeren satırlar:")
print(df[df["adet"] <= 0][["siparis_id", "urun", "adet"]])

df = df[df["adet"] > 0].copy()
print("\nYeni şekil:", df.shape)

Hatalı adet içeren satırlar:
     siparis_id            urun  adet
134        1135          Defter    -1
152        1153  Kahve Makinesi    -1
176        1177          Defter    -1

Yeni şekil: (297, 10)


### 6.4 Eksik değerler
Üç temel strateji vardır: **silmek**, **sabit bir değerle doldurmak**, **istatistiksel bir değerle (ortalama/medyan/mod) doldurmak**. Seçim, verinin anlamına bağlıdır.

In [20]:
print("Sütun bazında eksik değer sayısı:")
print(df.isna().sum())
print("\nEksik oranı (%):")
print((df.isna().mean() * 100).round(2))

Sütun bazında eksik değer sayısı:
siparis_id      0
tarih           0
musteri_id      0
il              8
urun            0
kategori        0
birim_fiyat     0
adet            0
odeme           0
puan           25
dtype: int64

Eksik oranı (%):
siparis_id     0.00
tarih          0.00
musteri_id     0.00
il             2.69
urun           0.00
kategori       0.00
birim_fiyat    0.00
adet           0.00
odeme          0.00
puan           8.42
dtype: float64


In [21]:
# il: kategorik değişken -> en sık görülen değer (mod) ya da "Bilinmiyor"
df["il"] = df["il"].fillna("Bilinmiyor")

# puan: sayısal değişken -> ürünün kendi medyan puanı ile doldur (daha akıllı yöntem)
df["puan"] = df["puan"].fillna(df.groupby("urun")["puan"].transform("median"))

print(df.isna().sum().sum(), "eksik değer kaldı")

0 eksik değer kaldı


> 💡 `dropna()` ile eksik satırları silmek en kolay yoldur ama veri kaybına yol açar. Eksik oranı çok düşükse (%1–2) silmek kabul edilebilir.

## 7. Yeni Sütun Türetme (Feature Engineering)

In [22]:
df["tutar"] = (df["birim_fiyat"] * df["adet"]).round(2)

# Tarihten bilgi çıkarma: .dt erişimcisi
df["ay"] = df["tarih"].dt.month
df["gun_adi"] = df["tarih"].dt.day_name()
df["hafta_sonu"] = df["tarih"].dt.dayofweek >= 5

# Koşullu sütun
df["kargo"] = np.where(df["tutar"] >= 1000, "Ücretsiz", "Ücretli")

df[["tarih", "ay", "gun_adi", "hafta_sonu", "tutar", "kargo"]].head()

,tarih,ay,gun_adi,hafta_sonu,tutar,kargo
0,2025-05-13,5,Tuesday,False,121864.92,Ücretsiz
1,2025-10-29,10,Wednesday,False,173.61,Ücretli
2,2025-01-04,1,Saturday,True,208.67,Ücretli
3,2025-11-24,11,Monday,False,2390.33,Ücretsiz
4,2025-07-18,7,Friday,False,9951.84,Ücretsiz


In [23]:
# pd.cut: sayısal değişkeni aralıklara bölme (ayrıklaştırma / binning)
df["tutar_grubu"] = pd.cut(
    df["tutar"],
    bins=[0, 500, 5000, 50000, np.inf],
    labels=["Düşük", "Orta", "Yüksek", "Çok Yüksek"],
)
df["tutar_grubu"].value_counts()

tutar_grubu
Orta          93
Yüksek        84
Düşük         80
Çok Yüksek    40
Name: count, dtype: int64

In [24]:
# map ve apply ile dönüşüm
mevsim = {12: "Kış", 1: "Kış", 2: "Kış", 3: "İlkbahar", 4: "İlkbahar", 5: "İlkbahar",
          6: "Yaz", 7: "Yaz", 8: "Yaz", 9: "Sonbahar", 10: "Sonbahar", 11: "Sonbahar"}
df["mevsim"] = df["ay"].map(mevsim)

def memnuniyet(puan):
    if puan >= 4:
        return "Memnun"
    elif puan == 3:
        return "Nötr"
    return "Memnun Değil"

df["memnuniyet"] = df["puan"].apply(memnuniyet)
df[["ay", "mevsim", "puan", "memnuniyet"]].sample(5, random_state=1)

,ay,mevsim,puan,memnuniyet
139,5,İlkbahar,5.0,Memnun
236,6,Yaz,5.0,Memnun
51,1,Kış,4.0,Memnun
293,12,Kış,5.0,Memnun
245,1,Kış,4.0,Memnun


### ✍️ Alıştırma 2
1. `df`'ye `kdv_dahil` adında, `tutar` değerinin %20 KDV eklenmiş halini içeren bir sütun ekleyin.
2. `ceyrek` adında, siparişin yılın kaçıncı çeyreğinde verildiğini gösteren bir sütun ekleyin. (İpucu: `.dt.quarter`)
3. Hafta sonu verilen siparişlerin toplam tutarını bulun.

In [25]:
# Kodunuzu buraya yazın

## 8. Sıralama

In [26]:
df.sort_values("tutar", ascending=False)[["siparis_id", "urun", "il", "tutar"]].head()

,siparis_id,urun,il,tutar
132,1133,Laptop,İstanbul,139145.76
271,1272,Laptop,Ankara,134390.24
278,1279,Laptop,Ankara,134105.88
68,1069,Laptop,İstanbul,131117.88
169,1170,Laptop,İstanbul,129354.20


In [27]:
# Birden fazla sütuna göre: önce kategori (A-Z), sonra tutar (büyükten küçüğe)
df.sort_values(["kategori", "tutar"], ascending=[True, False])[["kategori", "urun", "tutar"]].head(8)

,kategori,urun,tutar
132,Elektronik,Laptop,139145.76
271,Elektronik,Laptop,134390.24
278,Elektronik,Laptop,134105.88
68,Elektronik,Laptop,131117.88
169,Elektronik,Laptop,129354.20
126,Elektronik,Laptop,125302.52
6,Elektronik,Laptop,123666.84
0,Elektronik,Laptop,121864.92


## 9. Gruplama: `groupby` ⭐⭐
**Böl → Uygula → Birleştir (split-apply-combine)** mantığı. SQL'deki `GROUP BY` ile aynıdır.

In [28]:
# Kategori bazında toplam ciro
df.groupby("kategori")["tutar"].sum().sort_values(ascending=False)

kategori
Elektronik    4165832.20
Giyim          507313.17
Ev             281376.10
Kitap           16816.10
Kırtasiye       15847.34
Name: tutar, dtype: float64

In [29]:
# Birden fazla istatistik: agg
df.groupby("il").agg(
    siparis_sayisi=("siparis_id", "count"),
    toplam_ciro=("tutar", "sum"),
    ortalama_sepet=("tutar", "mean"),
    ortalama_puan=("puan", "mean"),
).round(1).sort_values("toplam_ciro", ascending=False)

,siparis_sayisi,toplam_ciro,ortalama_sepet,ortalama_puan
il,,,,
İstanbul,82,1341892.1,16364.5,3.9
Ankara,68,959189.9,14105.7,3.9
İzmir,46,750804.3,16321.8,4.1
Bursa,25,509502.1,20380.1,4.4
Gümüşhane,16,498309.9,31144.4,3.6
Erzurum,26,385865.3,14841.0,3.9
Trabzon,26,299449.2,11517.3,4.0
Bilinmiyor,8,242172.0,30271.5,4.0


In [30]:
# Birden fazla sütuna göre gruplama
df.groupby(["kategori", "odeme"])["tutar"].mean().round(0).unstack()

odeme,Havale,Kapıda Ödeme,Kredi Kartı
kategori,,,
Elektronik,37066.0,26338.0,57893.0
Ev,12825.0,17578.0,11823.0
Giyim,2549.0,5014.0,5507.0
Kitap,488.0,543.0,528.0
Kırtasiye,264.0,185.0,289.0


## 10. Pivot Tablo ve Çapraz Tablo
Excel'deki Özet Tablo (Pivot Table) özelliğinin aynısıdır.

In [31]:
pd.pivot_table(
    df, values="tutar", index="kategori", columns="mevsim",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Toplam"
).round(0)

mevsim,Kış,Sonbahar,Yaz,İlkbahar,Toplam
kategori,,,,,
Elektronik,1354427.0,1153796.0,427461.0,1230148.0,4165832.0
Ev,64196.0,73864.0,76441.0,66874.0,281376.0
Giyim,144598.0,70808.0,142268.0,149640.0,507313.0
Kitap,5226.0,5668.0,1965.0,3957.0,16816.0
Kırtasiye,5850.0,1630.0,4020.0,4348.0,15847.0
Toplam,1574297.0,1305766.0,652155.0,1454967.0,4987185.0


In [32]:
# crosstab: iki kategorik değişkenin frekans tablosu
pd.crosstab(df["kategori"], df["memnuniyet"], normalize="index").round(2)

memnuniyet,Memnun,Memnun Değil,Nötr
kategori,,,
Elektronik,0.77,0.06,0.17
Ev,0.77,0.09,0.14
Giyim,0.66,0.10,0.24
Kitap,0.78,0.03,0.19
Kırtasiye,0.72,0.08,0.20


### ✍️ Alıştırma 3
1. Her ürün için satılan **toplam adet** ve **ortalama puan**ı tek tabloda gösterin; toplam adede göre büyükten küçüğe sıralayın.
2. Aylara göre toplam ciroyu hesaplayın. En yüksek ciro hangi ayda gerçekleşmiş?
3. Satırlarda `il`, sütunlarda `kategori` olan ve hücrelerde **sipariş sayısını** gösteren bir pivot tablo oluşturun.

In [33]:
# Kodunuzu buraya yazın

## 11. Tabloları Birleştirme: `merge` ve `concat`
Gerçek projelerde veri tek tabloda gelmez. Örneğin müşteri bilgileri ayrı bir tabloda tutulur.

In [34]:
rng2 = np.random.default_rng(1)
musteriler = pd.DataFrame({
    "musteri_id": np.arange(1, 76),     # dikkat: 76-80 arası müşteriler bu tabloda yok!
    "cinsiyet": rng2.choice(["K", "E"], size=75),
    "yas": rng2.integers(18, 65, size=75),
    "uyelik": rng2.choice(["Standart", "Premium"], size=75, p=[0.7, 0.3]),
})
musteriler.head()

,musteri_id,cinsiyet,yas,uyelik
0,1,K,42,Standart
1,2,E,58,Standart
2,3,E,39,Premium
3,4,E,35,Premium
4,5,K,20,Premium


In [35]:
inner = df.merge(musteriler, on="musteri_id", how="inner")
left  = df.merge(musteriler, on="musteri_id", how="left")

print("Sipariş tablosu  :", df.shape)
print("inner join sonucu:", inner.shape, "-> eşleşmeyenler atıldı")
print("left join sonucu :", left.shape, "-> tüm siparişler korundu, eşleşmeyenlerde NaN")
print("Eşleşmeyen sipariş:", left["uyelik"].isna().sum())

Sipariş tablosu  : (297, 18)
inner join sonucu: (278, 21) -> eşleşmeyenler atıldı
left join sonucu : (297, 21) -> tüm siparişler korundu, eşleşmeyenlerde NaN
Eşleşmeyen sipariş: 19


In [36]:
# Birleştirilmiş veriyle analiz: üyelik tipine göre ortalama sepet tutarı
inner.groupby("uyelik")["tutar"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
uyelik,,,
Premium,82,10495.0,2708.0
Standart,196,20193.0,3316.0


In [37]:
# concat: aynı yapıdaki tabloları alt alta ekleme
ocak = df[df["ay"] == 1]
subat = df[df["ay"] == 2]
ilk_iki_ay = pd.concat([ocak, subat], ignore_index=True)
print(len(ocak), "+", len(subat), "=", len(ilk_iki_ay))

24 + 33 = 57


## 12. Zaman Serisi Özetleme ve Dosyaya Kaydetme

In [38]:
aylik = df.set_index("tarih").resample("ME")["tutar"].sum()   # ME = ay sonu
aylik.index = aylik.index.strftime("%Y-%m")
aylik.round(0)

tarih
2025-01    506514.0
2025-02    411018.0
2025-03    325133.0
2025-04    718622.0
2025-05    411212.0
2025-06    326063.0
2025-07     88991.0
2025-08    237101.0
2025-09    187051.0
2025-10    550926.0
2025-11    567789.0
2025-12    656765.0
Name: tutar, dtype: float64

In [39]:
# Temizlenmiş veriyi kaydetme (sonraki defterlerde kullanabilirsiniz)
df.to_csv("eticaret_temiz.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: Excel'de Türkçe karakterler düzgün görünür
kontrol = pd.read_csv("eticaret_temiz.csv", parse_dates=["tarih"])
print("Kaydedildi ve tekrar okundu:", kontrol.shape)

Kaydedildi ve tekrar okundu: (297, 18)


### ✍️ Alıştırma 4 (Kapsamlı): RFM Analizi
RFM, müşteri segmentasyonunda kullanılan klasik bir veri madenciliği yöntemidir:
- **R (Recency):** Müşterinin son alışverişinden bu yana geçen gün sayısı (analiz tarihi: `2026-01-01`)
- **F (Frequency):** Toplam sipariş sayısı
- **M (Monetary):** Toplam harcama tutarı

Görevler:
1. `df`'yi `musteri_id`'ye göre gruplayarak her müşteri için R, F, M değerlerini içeren `rfm` tablosunu oluşturun.
2. M değerine göre en değerli 5 müşteriyi listeleyin.
3. `pd.qcut` kullanarak M değerini 3 eşit gruba bölün (`"Bronz"`, `"Gümüş"`, `"Altın"`) ve her grupta kaç müşteri olduğunu gösterin.

In [40]:
# Kodunuzu buraya yazın

---
## 📌 Özet
| Adım | Komutlar |
|---|---|
| Okuma / yazma | `read_csv`, `read_excel`, `to_csv` |
| İlk keşif | `head`, `info`, `describe`, `value_counts`, `nunique` |
| Seçim | `df[...]`, `loc`, `iloc`, `query`, `isin`, `.str.contains` |
| Temizlik | `duplicated`, `drop_duplicates`, `isna`, `fillna`, `dropna`, `.str.strip` |
| Türetme | aritmetik, `np.where`, `pd.cut`, `map`, `apply`, `.dt` |
| Özetleme | `groupby` + `agg`, `pivot_table`, `crosstab`, `resample` |
| Birleştirme | `merge (inner/left/right/outer)`, `concat` |

**Sonraki defter:** `03_matplotlib.ipynb` — veriyi görselleştirme.

---
## ✅ Çözümler

### Çözüm 1

In [41]:
print(df[["urun", "kategori", "puan"]].head())
print("\nKapıda ödeme & puan <= 2:", len(df[(df["odeme"] == "Kapıda Ödeme") & (df["puan"] <= 2)]))
df[df["kategori"].isin(["Giyim", "Kitap"]) & (df["birim_fiyat"] < 1000)].head()

       urun    kategori  puan
0    Laptop  Elektronik   5.0
1    Defter   Kırtasiye   5.0
2     Kitap       Kitap   4.0
3  Ayakkabı       Giyim   4.0
4  Ayakkabı       Giyim   5.0

Kapıda ödeme & puan <= 2: 3


,siparis_id,tarih,musteri_id,il,urun,kategori,birim_fiyat,adet,odeme,puan,tutar,ay,gun_adi,hafta_sonu,kargo,tutar_grubu,mevsim,memnuniyet
2,1003,2025-01-04,60,Trabzon,Kitap,Kitap,208.67,1,Havale,4.0,208.67,1,Saturday,True,Ücretli,Düşük,Kış,Memnun
7,1008,2025-07-22,42,Ankara,Kitap,Kitap,238.94,1,Kredi Kartı,4.0,238.94,7,Tuesday,False,Ücretli,Düşük,Yaz,Memnun
10,1011,2025-09-01,67,İzmir,Tişört,Giyim,469.04,2,Havale,3.0,938.08,9,Monday,False,Ücretli,Orta,Sonbahar,Nötr
16,1017,2025-09-24,20,İzmir,Tişört,Giyim,421.95,4,Kredi Kartı,2.0,1687.80,9,Wednesday,False,Ücretsiz,Orta,Sonbahar,Memnun Değil
20,1021,2025-04-28,35,Ankara,Tişört,Giyim,426.75,4,Kredi Kartı,3.0,1707.00,4,Monday,False,Ücretsiz,Orta,İlkbahar,Nötr


### Çözüm 2

In [42]:
df["kdv_dahil"] = (df["tutar"] * 1.20).round(2)
df["ceyrek"] = df["tarih"].dt.quarter
print("Hafta sonu toplam tutar:", df.loc[df["hafta_sonu"], "tutar"].sum().round(2))
df[["tutar", "kdv_dahil", "tarih", "ceyrek"]].head()

Hafta sonu toplam tutar: 1862932.6


,tutar,kdv_dahil,tarih,ceyrek
0,121864.92,146237.90,2025-05-13,2
1,173.61,208.33,2025-10-29,4
2,208.67,250.40,2025-01-04,1
3,2390.33,2868.40,2025-11-24,4
4,9951.84,11942.21,2025-07-18,3


### Çözüm 3

In [43]:
print(df.groupby("urun").agg(toplam_adet=("adet", "sum"), ortalama_puan=("puan", "mean"))
        .round(2).sort_values("toplam_adet", ascending=False))

aylik_ciro = df.groupby("ay")["tutar"].sum()
print("\nEn yüksek cirolu ay:", aylik_ciro.idxmax(), "->", aylik_ciro.max().round(0), "TL")

pd.pivot_table(df, values="siparis_id", index="il", columns="kategori", aggfunc="count", fill_value=0)

                toplam_adet  ortalama_puan
urun                                      
Ayakkabı                125           3.83
Defter                   95           3.97
Laptop                   82           4.07
Kitap                    75           4.03
Kulaklık                 74           3.97
Tişört                   61           4.00
Kahve Makinesi           59           4.36
Kalem Seti               57           3.91
Mont                     57           3.92
Telefon                  56           3.92

En yüksek cirolu ay: 4 -> 718622.0 TL


kategori,Elektronik,Ev,Giyim,Kitap,Kırtasiye
il,,,,,
Ankara,17,2,25,10,14
Bilinmiyor,3,1,3,1,0
Bursa,5,4,8,2,6
Erzurum,7,2,8,1,8
Gümüşhane,9,1,3,3,0
Trabzon,6,2,8,4,6
İstanbul,22,6,28,9,17
İzmir,14,4,17,2,9


### Çözüm 4

In [44]:
analiz_tarihi = pd.Timestamp("2026-01-01")
rfm = df.groupby("musteri_id").agg(
    R=("tarih", lambda t: (analiz_tarihi - t.max()).days),
    F=("siparis_id", "count"),
    M=("tutar", "sum"),
)
print(rfm.head())
print("\nEn değerli 5 müşteri:")
print(rfm.sort_values("M", ascending=False).head().round(0))

rfm["segment"] = pd.qcut(rfm["M"], q=3, labels=["Bronz", "Gümüş", "Altın"])
print("\n", rfm["segment"].value_counts())

             R  F         M
musteri_id                 
1           62  6  76435.79
2           60  3   6303.50
3           29  4  15589.14
4            2  3  85711.67
5           23  6  22079.70

En değerli 5 müşteri:
             R  F         M
musteri_id                 
74          49  5  271734.0
44          28  4  252614.0
50          42  7  228182.0
58          50  5  189299.0
55          28  6  184515.0

 segment
Bronz    26
Gümüş    26
Altın    26
Name: count, dtype: int64
